#Chroma DB Persistence


In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import tempfile
import shutil

load_dotenv()

embeddings_model=OpenAIEmbeddings(model="text-embedding-3-small")

SAMPLE_DOCS=[
    Document(page_content="LangChain is a framework for building applications powered by LLMs by connecting models with prompts, tools, data, memory, and retrieval systems.",
    metadata={"source": "langchain_docs", "topic": "overview"},),

    Document(page_content="LangGraph is a framework for building stateful, multi-step agent workflows with LLMs.",
    metadata={"source": "langgraph_docs", "topic": "overview"},),

     Document(page_content="Vector stores are databases optimized for storing and seraching embeddings.",
    metadata={"source": "vector_guide", "topic": "database"},),

     Document(page_content="RAG combines retrieval with generation for more accurate LLM response.",
    metadata={"source": "rag_guide", "topic": "architecture"},),

    Document(page_content="Embeddings are numerical vector representations of text that capture semantic meaning.",
    metadata={"source": "embedding_guide", "topic": "embeddings"},),

    Document(page_content="Chroma is an open-source vector database designed for storing and searching embeddings.",
    metadata={"source": "Chroma_docs", "topic": "database"},),

    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.",
    metadata={"source": "faiss_docs", "topic": "database"},),

     Document(page_content="Pinecone provides a managed vector database for storing, indexing, and searching vector embeddings in production applications.",
    metadata={"source": "pinecone_docs", "topic": "database"},),

]

def persist_chroma():
    persist_dir = "./chroma_db/"

    vectorstore = Chroma.from_documents(
        documents=SAMPLE_DOCS,
        embedding=embeddings_model,
        persist_directory=persist_dir,
    )
    #Saved inside chroma.sqlite3

    #Internally this is like:
    # for doc in SAMPLE_DOCS:
    #    vector = embeddings_model.embed_documents([doc.page_content])

    #    save_to_chroma(doc, vector)

    #By Python convention, a name starting with _ is considered an internal implementation detail.
    original_count = vectorstore._collection.count()

    print(f"Persisted vector store with {original_count} documents.")
    print(f"Vector store persisted at: {persist_dir}")

    #simulate restart - load from disk
    del vectorstore #does not delete the vector database.
    #Delete the variable (reference) from the current namespace.
    #Delete the variable (reference) from the current namespace.

    reloaded=Chroma(
             embedding_function=embeddings_model, #Embedding Function->Query Vector->Compare against stored vectors
             persist_directory=persist_dir,)

    reloaded_count = reloaded._collection.count()
    print(f"Reloaded vector store with {reloaded_count} documents.")
    # verify search still works
    results = reloaded.similarity_search("LangChain", k=2)
    print(f"Search result: {results[0].page_content[:50]}...")

if __name__ == "__main__":
    persist_chroma()

#Vector Store as retriever for chains

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
import tempfile

embeddings_model=OpenAIEmbeddings(model="text-embedding-3-small")

SAMPLE_DOCS=[
    Document(page_content="LangChain is a framework for building applications powered by LLMs by connecting models with prompts, tools, data, memory, and retrieval systems.",
    metadata={"source": "langchain_docs", "topic": "overview"},),

    Document(page_content="LangGraph is a framework for building stateful, multi-step agent workflows with LLMs.",
    metadata={"source": "langgraph_docs", "topic": "overview"},),

     Document(page_content="Vector stores are databases optimized for storing and seraching embeddings.",
    metadata={"source": "vector_guide", "topic": "database"},),

     Document(page_content="RAG combines retrieval with generation for more accurate LLM response.",
    metadata={"source": "rag_guide", "topic": "architecture"},),

    Document(page_content="Embeddings are numerical vector representations of text that capture semantic meaning.",
    metadata={"source": "embedding_guide", "topic": "embeddings"},),

    Document(page_content="Chroma is an open-source vector database designed for storing and searching embeddings.",
    metadata={"source": "Chroma_docs", "topic": "database"},),

    Document(page_content="FAISS is a library for efficient similarity search and clustering of dense vectors.",
    metadata={"source": "faiss_docs", "topic": "database"},),

     Document(page_content="Pinecone provides a managed vector database for storing, indexing, and searching vector embeddings in production applications.",
    metadata={"source": "pinecone_docs", "topic": "database"},),

]

def demo_vectorStoreAsRetriever():
    with tempfile.TemporaryDirectory() as tmpdir:
        # create vector store from documents
        vectorstore=Chroma.from_documents(
        documents=SAMPLE_DOCS,
        embedding=embeddings_model,
        persist_directory=tmpdir,
        )

        #Basic retriever usage
        #This does not perform a search.It simply wraps the vector store inside a Retriever object.
        #returns a VectorStoreRetriever object, not documents.
        retriever=vectorstore.as_retriever(
            search_type="similarity", #Other search types include Maximum Marginal Relevance,similarity_score_threshold
            search_kwargs={"k":3}
        )
        #LangChain prefers using retrievers because they provide a standard interface.Chroma,Pinecone,FAISS,Milvus all expose the same retriever API.
        #So later your code becomes independent of the underlying vector database.
        #we can call invoke on it, as it is a chain (runnable)This is because VectorStoreRetriever implements LangChain's Runnable interface.A Runnable is anything that accepts an input and produces an output.


        #Maximal Marginal Relevance (MMR) 
        mmr_retriever=vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 3, "fetch_k": 5} # First retrieve the top 5 most similar documents and return 3 diverse.
        )
        #Return documents that are both relevant to the query and different from each other.That way The context is richer and less repetitive.
        mmr_docs = mmr_retriever.invoke("How do I build AI applications?")

        #USE THE RETRIEVER TO GET RELEVANT DOCUMENTS
        docs=retriever.invoke("How do I build AI Applications?")

        print("Retriver Results:")
        for i,doc in enumerate(docs):
            print(f"Result {i+1}: Content: {doc.page_content} Source: {doc.metadata["source"]}")
if __name__ =="__main__":
    demo_vectorStoreAsRetriever()


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\DELL\\AppData\\Local\\Temp\\tmp9zg6fo7g\\chroma.sqlite3'